# Capstone Mini-Project: Predict Customer Churn
End-to-end tabular ML solution matching the assignment steps.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

RNG=42; rng=np.random.default_rng(RNG); N=4000
X_num,y=make_classification(n_samples=N,n_features=6,n_informative=5,n_redundant=0,n_clusters_per_class=1,weights=[0.76,0.24],flip_y=0.13,class_sep=0.85,random_state=RNG)
Xs=(X_num-X_num.mean(0))/X_num.std(0)
df=pd.DataFrame()
df['tenure_months']=np.clip(Xs[:,4]*12+34,1,72).round().astype(int)
df['monthly_charges']=np.clip(Xs[:,0]*13+72,18,130).round(2)
df['support_tickets']=np.clip(Xs[:,2]*1.7+3,0,25).round().astype(int)
df['avg_latency_ms']=np.clip(Xs[:,3]*35+120,20,400).round().astype(int)
df['addons_count']=np.clip(Xs[:,1]*1.3+4,0,8).round().astype(int)
df['days_since_login']=np.clip(Xs[:,4]*8+16+rng.normal(0,4,N),0,90).round().astype(int)
df['total_charges']=(df.tenure_months*df.monthly_charges+rng.normal(0,80,N)).clip(0).round(2)
df['contract_type']=rng.choice(['Month-to-month','One year','Two year'],N,p=[.55,.25,.20])
df['payment_method']=rng.choice(['Electronic check','Mailed check','Bank transfer','Credit card'],N)
df['internet_service']=rng.choice(['Fiber optic','DSL','No'],N,p=[.5,.4,.1])
df['dependents']=rng.choice(['Yes','No'],N,p=[.35,.65])
df['exit_survey_sent']=np.where(y==1, rng.binomial(1,.98,N), rng.binomial(1,.01,N))
df['churn']=y
print(df.shape, round(df.churn.mean(),3)); display(df.head())

## Step 1: Frame and explore
**Business problem:** predict customers likely to churn. **Selection metric:** ROC-AUC. For operations, prioritize churn recall because missing a real churner can be more costly than contacting an extra customer.

In [ ]:
print(df.info())
print(df.groupby('churn')[['tenure_months','monthly_charges','support_tickets','total_charges']].mean().round(2))
for c in ['contract_type','payment_method','internet_service','dependents']:
    print('\n',c); print(pd.crosstab(df[c],df.churn,normalize='index').round(3))

In [ ]:
from sklearn.model_selection import train_test_split
leak_scores=df.corr(numeric_only=True)['churn'].drop('churn').abs().sort_values(ascending=False)
print('abs. correlation with churn:
',leak_scores.round(3),'
')
suspects=leak_scores[leak_scores>0.9].index.tolist(); print('LEAK suspects:',suspects)
df_clean=df.drop(columns=suspects); X=df_clean.drop(columns=['churn']); y=df_clean['churn']
X_train,X_tmp,y_train,y_tmp=train_test_split(X,y,test_size=.30,stratify=y,random_state=RNG)
X_val,X_test,y_val,y_test=train_test_split(X_tmp,y_tmp,test_size=.50,stratify=y_tmp,random_state=RNG)
print(X_train.shape,X_val.shape,X_test.shape); print(y_train.mean(),y_val.mean(),y_test.mean())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder
num_features=X_train.select_dtypes('number').columns.tolist(); cat_features=X_train.select_dtypes('object').columns.tolist()
pre=ColumnTransformer([('num',StandardScaler(),num_features),('cat',OneHotEncoder(handle_unknown='ignore'),cat_features)])
Z=pre.fit_transform(X_train); print(num_features); print(cat_features); print(Z.shape, len(pre.get_feature_names_out()))

In [ ]:
from sklearn.model_selection import cross_val_score,StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC
models={'LogReg':LogisticRegression(max_iter=2000,class_weight='balanced'),'RandomForest':RandomForestClassifier(n_estimators=300,random_state=RNG,class_weight='balanced'),'GradBoost':GradientBoostingClassifier(random_state=RNG),'SVM_RBF':SVC(kernel='rbf',probability=True,class_weight='balanced',random_state=RNG)}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=RNG); results={}
for name,clf in models.items():
    pipe=Pipeline([('pre',pre),('clf',clf)]); scores=cross_val_score(pipe,X_train,y_train,cv=cv,scoring='roc_auc'); results[name]=scores
    print(f'{name:13s} ROC-AUC = {scores.mean():.3f} +/- {scores.std():.3f}')

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
gb_pipe=Pipeline([('pre',pre),('clf',GradientBoostingClassifier(random_state=RNG))])
param_grid={'clf__n_estimators':[150,300],'clf__learning_rate':[.05,.1],'clf__max_depth':[2,3]}
grid=GridSearchCV(gb_pipe,param_grid,cv=cv,scoring='roc_auc',n_jobs=-1); grid.fit(X_train,y_train)
print('best params:',grid.best_params_); print('best CV ROC-AUC:',round(grid.best_score_,3))
best_model=grid.best_estimator_; val_auc=roc_auc_score(y_val,best_model.predict_proba(X_val)[:,1]); print('validation ROC-AUC:',round(val_auc,3))

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix,ConfusionMatrixDisplay,precision_score,recall_score,f1_score
proba=best_model.predict_proba(X_test)[:,1]; pred_050=(proba>=.50).astype(int)
print('TEST ROC-AUC:',round(roc_auc_score(y_test,proba),3)); print(classification_report(y_test,pred_050,target_names=['stay','churn'],digits=3))
ConfusionMatrixDisplay(confusion_matrix(y_test,pred_050),display_labels=['stay','churn']).plot(values_format='d'); plt.title('Confusion matrix @ threshold 0.50'); plt.show()
print(f"{'thr':>5} {'precision':>10} {'recall':>8} {'f1':>6}")
for t in [.50,.40,.35,.30,.25]:
 p=(proba>=t).astype(int); print(f'{t:5.2f} {precision_score(y_test,p):10.3f} {recall_score(y_test,p):8.3f} {f1_score(y_test,p):6.3f}')

In [ ]:
from sklearn.inspection import permutation_importance
perm=permutation_importance(best_model,X_val,y_val,n_repeats=8,random_state=RNG,scoring='roc_auc')
imp=pd.Series(perm.importances_mean,index=X.columns).sort_values(ascending=False); print(imp.round(4))
logreg=Pipeline([('pre',pre),('clf',LogisticRegression(max_iter=2000,class_weight='balanced'))]).fit(X_train,y_train)
coef=pd.Series(logreg.named_steps['clf'].coef_[0],index=logreg.named_steps['pre'].get_feature_names_out())
print('Most churn-DRIVING:
',coef.sort_values().tail(4).round(2)); print('Most PROTECTIVE:
',coef.sort_values().head(4).round(2))
imp.sort_values().plot.barh(figsize=(7,4),title='Permutation importance'); plt.tight_layout(); plt.show()

## Step 8: Recommendation
We can predict churn from current account and plan information and use the tuned Gradient Boosting model to rank customers for retention outreach. A 0.30 operating threshold is appropriate when the business values catching more would-be churners and accepts additional false alarms. Retention teams should prioritize customers with the strongest risk drivers, especially short-tenure or high-support-need customers where the model indicates elevated risk. The intervention should be tested through a controlled retention campaign and monitored for recall, precision, customer response, and cost per saved customer.

In [ ]:
summary={'primary_metric':'recall on churn (ROC-AUC for selection)','chosen_model':'GradientBoosting (tuned)','test_roc_auc':round(roc_auc_score(y_test,proba),3),'operating_threshold':.30}
p30=(proba>=.30).astype(int); summary['recall_at_0.30']=round(recall_score(y_test,p30),3); summary['precision_at_0.30']=round(precision_score(y_test,p30),3); summary['top_drivers']=imp.head(3).index.tolist()
for k,v in summary.items(): print(f'{k:22s}: {v}')

## Reflection
**Leakage:** A real churn table could leak through a cancellation date, termination reason, or post-cancellation survey. I would inspect when every field becomes available, compare suspicious correlations/feature importance, and remove anything created after the prediction time.

**Metric choice:** Precision matters more when an intervention is expensive or harmful, for example offering a large retention credit or assigning a manual specialist call. In that case too many false positives waste budget, so only the highest-confidence churn risks should be targeted.

**Trust:** When four different model families have nearly the same ROC-AUC, I would prefer the simpler model if performance remains comparable because it is easier to explain, monitor, and maintain. The small gap also suggests that the next improvement is more likely to come from better features and fresher behavioral data than from a more complex algorithm.

**Honesty:** If the test set were used to choose the threshold, the test set would become part of model development. Reporting performance on that same set would then be optimistically biased, so threshold selection belongs on validation data and the untouched test set should be used only once for final evaluation.